# Тесты

Проверка корректности реализованных классов и методов.

## Блок 1: Конструктивное число

In [ ]:
from src.ConstructiveNumber import ConstructiveNumber as CN

# Создание из пары (a, b)
c1 = CN(1.0, 2.0)
assert c1.a == 1.0 and c1.b == 2.0
assert c1.epsilon == 1.0

# Создание из (x, epsilon)
c2 = CN(5.0, epsilon=0.1)
assert abs(c2.a - 4.95) < 1e-12
assert abs(c2.b - 5.05) < 1e-12
assert abs(c2.epsilon - 0.1) < 1e-12

# Получение значения через alpha
assert c1.value(0) == 1.0
assert c1.value(1) == 2.0
assert c1.value(0.5) == 1.5

print('Создание и value: OK')

In [ ]:
# Сложение CN + CN
a = CN(1.0, 2.0)
b = CN(3.0, 4.0)
s = a + b
assert s.a == 4.0 and s.b == 6.0

# Сложение CN + float
s2 = a + 5.0
assert s2.a == 6.0 and s2.b == 7.0

# float + CN
s3 = 5.0 + a
assert s3.a == 6.0 and s3.b == 7.0

print('Сложение: OK')

In [ ]:
# Вычитание
a = CN(3.0, 5.0)
b = CN(1.0, 2.0)
d = a - b
assert d.a == 1.0 and d.b == 4.0  # [3-2, 5-1]

# CN - float
d2 = a - 1.0
assert d2.a == 2.0 and d2.b == 4.0

# float - CN
d3 = 10.0 - a
assert d3.a == 5.0 and d3.b == 7.0  # [10-5, 10-3]

print('Вычитание: OK')

In [ ]:
# Умножение
a = CN(2.0, 3.0)
b = CN(4.0, 5.0)
m = a * b
assert m.a == 8.0 and m.b == 15.0

# Умножение с отрицательными
a = CN(-1.0, 2.0)
b = CN(3.0, 4.0)
m = a * b
# products: -3, -4, 6, 8
assert m.a == -4.0 and m.b == 8.0

# CN * float
a = CN(1.0, 3.0)
m2 = a * 2.0
assert m2.a == 2.0 and m2.b == 6.0

print('Умножение: OK')

In [ ]:
# Деление
a = CN(4.0, 6.0)
b = CN(2.0, 3.0)
d = a / b
# a * [1/3, 1/2] => products: 4/3, 4/2, 6/3, 6/2 => min=4/3, max=3
assert abs(d.a - 4.0/3.0) < 1e-12
assert abs(d.b - 3.0) < 1e-12

# Деление на ноль
try:
    a / CN(-1.0, 1.0)
    assert False, 'Должно было быть исключение'
except ZeroDivisionError:
    pass

print('Деление: OK')

In [ ]:
# Возведение в степень
a = CN(2.0, 3.0)
p = a ** 2
assert p.a == 4.0 and p.b == 9.0

a = CN(-3.0, -1.0)
p = a ** 2
assert p.a == 1.0 and p.b == 9.0

a = CN(-2.0, 3.0)
p = a ** 2
assert p.a == 0.0 and p.b == 9.0

print('Степень: OK')

In [ ]:
# Сравнение
a = CN(1.0, 2.0)
b = CN(3.0, 4.0)
assert a < b
assert b > a
assert not (a > b)

# Равенство
assert CN(1.0, 2.0) == CN(1.0, 2.0)
assert not (CN(1.0, 2.0) == CN(1.0, 3.0))

print('Сравнение: OK')

## Блок 2: Чёрные ящики

In [ ]:
import numpy as np
from src.BlackBox import QuadraticGoodCond, QuadraticBadCond, Rosenbrock

# Квадратичная с хорошей обусловленностью
f1 = QuadraticGoodCond()
assert f1.dim == 6
x_min = f1.minimum
g_at_min = f1.grad(x_min)
assert np.linalg.norm(g_at_min) < 1e-10, f'Градиент в минимуме не ноль: {np.linalg.norm(g_at_min)}'
print(f'{f1.name}: dim={f1.dim}, ||grad(x*)||={np.linalg.norm(g_at_min):.2e}')

# Квадратичная с плохой обусловленностью
f2 = QuadraticBadCond()
assert f2.dim == 4
x_min = f2.minimum
g_at_min = f2.grad(x_min)
assert np.linalg.norm(g_at_min) < 1e-10
print(f'{f2.name}: dim={f2.dim}, ||grad(x*)||={np.linalg.norm(g_at_min):.2e}')

# Розенброк
f3 = Rosenbrock()
assert f3.dim == 3
x_min = f3.minimum
assert abs(f3.func(x_min)) < 1e-12
assert np.linalg.norm(f3.grad(x_min)) < 1e-10
print(f'{f3.name}: dim={f3.dim}, f(x*)={f3.func(x_min):.2e}')

print('\nВсе чёрные ящики: OK')

In [ ]:
# Проверка совместимости с конструктивными числами
from src.ConstructiveNumber import ConstructiveNumber as CN

f1 = QuadraticGoodCond()
f1.reset_counters()

x_cn = [CN(0.0, epsilon=0.01) for _ in range(f1.dim)]
val = f1.func(x_cn)
grad = f1.grad(x_cn)

print(f'f(0) с CN: {val}')
print(f'epsilon f: {val.epsilon:.6f}')
print(f'grad[0]: {grad[0]}')
print(f'Вызовов функции: {f1.func_calls}, градиента: {f1.grad_calls}')

print('\nСовместимость CN + BlackBox: OK')

## Блок 3: Методы оптимизации

In [ ]:
from src.Optimizers import gradient_descent, nelder_mead, newton_method

# Быстрый тест на простой квадратичной функции
f1 = QuadraticGoodCond()
x0 = np.zeros(f1.dim)

# Градиентный спуск
f1.reset_counters()
x_gd, h_gd = gradient_descent(f1, x0, lr=0.5, max_iter=500)
print(f'GD: ||x - x*|| = {np.linalg.norm(x_gd - f1.minimum):.2e}, iters={len(h_gd["f"])}, f_calls={f1.func_calls}')

# Нелдер-Мид
f1.reset_counters()
x_nm, h_nm = nelder_mead(f1, x0, max_iter=2000)
print(f'NM: ||x - x*|| = {np.linalg.norm(x_nm - f1.minimum):.2e}, iters={len(h_nm["f"])}, f_calls={f1.func_calls}')

# Метод Ньютона
f1.reset_counters()
x_nt, h_nt = newton_method(f1, x0, max_iter=100)
print(f'Newton: ||x - x*|| = {np.linalg.norm(x_nt - f1.minimum):.2e}, iters={len(h_nt["f"])}, f_calls={f1.func_calls}')

print('\nМетоды оптимизации: OK')